# 2.1 마르코프 의사결정 과정

MDP(Markov Decision Process)는 강화학습의 수학적 기초이다. LLM 에이전트도 이 프레임워크로 모델링할 수 있다.

### MDP의 다섯 가지 요소
- **S (상태 공간)**: 에이전트가 관찰할 수 있는 모든 가능한 상태
- **A (행동 공간)**: 에이전트가 선택할 수 있는 모든 행동
- **T (전이 함수)**: T(s'|s,a) = 상태 s에서 행동 a를 취할 때 상태 s'로 전이될 확률
- **R (보상 함수)**: R(s,a,s') = 상태 s에서 행동 a를 취해 s'로 전이될 때 받는 즉각적인 보상
- **γ (할인 인수)**: 0 < γ ≤ 1, 미래 보상의 현재 가치를 결정한다

### 마르코프 성질
미래는 오직 현재 상태에만 의존하며, 과거는 무관하다.
$P(s_{t+1} | s_t, a_t) = P(s_{t+1} | h_t) (h_t는 전체 히스토리)$

## LLM 에이전트와 MDP

### Agent RL에서 MDP 구조
LLM 에이전트도 MDP로 모델링할 수 있다:
- **S**: 텍스트 상태 (사용자 프롬프트, 대화 히스토리, 환경 정보)
- **A**: LLM의 생성 출력 (텍스트 응답, 도구 호출)
- **T**: LLM의 조건부 분포 p(next_state | current_state, action)
- **R**: 외부 평가 신호 (사용자 피드백, 작업 완료 신호)
- **γ**: 길고 반복적인 대화를 고려한 할인

### 결정적 vs 확률적 MDP
- **결정적**: T(s'|s,a) = 0 or 1 (temperature=0일 때 LLM)
- **확률적**: T(s'|s,a) = p (0 < p < 1) (temperature>0일 때 LLM)

In [2]:
from utils_openai import *

## 실습 1: 텍스트 기반 상태 공간 정의

In [3]:
heading("텍스트 상태 공간 구현")

# LLM 에이전트의 상태는 텍스트로 표현된다
class TextState:
    def __init__(self, user_input, history=None, context=""):
        self.user_input = user_input  # 현재 사용자 입력
        self.history = history or []   # 대화 히스토리
        self.context = context         # 환경 맥락 정보
    
    def __repr__(self):
        return f"TextState(input='{self.user_input[:30]}...', history_len={len(self.history)})"

# 초기 상태 생성
initial_state = TextState(
    user_input="2+2는 몇인가?",
    history=[],
    context="수학 문제 풀이 모드"
)

step_print(1, "초기 상태", str(initial_state))
kv_print([
    ("S (상태 공간)", "텍스트로 표현된 에이전트의 인식"),
    ("사용자 입력", initial_state.user_input),
    ("히스토리 길이", len(initial_state.history)),
    ("환경 맥락", initial_state.context)
])
print("\
✓ LLM 에이전트의 상태는 텍스트 및 메타데이터로 구성된다.")


────────────────────────────────────────
  텍스트 상태 공간 구현
────────────────────────────────────────
  [Step 1] 초기 상태
    TextState(input='2+2는 몇인가?...', history_len=0)
  S (상태 공간)  텍스트로 표현된 에이전트의 인식
  사용자 입력     2+2는 몇인가?
  히스토리 길이    0
  환경 맥락      수학 문제 풀이 모드
✓ LLM 에이전트의 상태는 텍스트 및 메타데이터로 구성된다.


## 실습 2: 행동 공간과 전이 함수

In [4]:
heading("텍스트 상태 전이 시뮬레이션")

# 현재 상태에서 행동을 취한다 (ask() 함수 사용)
current_state = TextState(
    user_input="2+2는 몇인가?",
    history=[],
    context="수학 문제"
)

step_print(1, "현재 상태", current_state.user_input)

# 행동: 에이전트가 응답 생성 (LLM 호출)
step_print(2, "행동 (A)", "LLM 기반 응답 생성")
action_response = ask(current_state.user_input, temperature=0.0)  # 결정적 전이
step_print(3, "응답", action_response)

# 새로운 상태로 전이
next_state = TextState(
    user_input="응답이 맞는가?",
    history=[{"user": current_state.user_input, "assistant": action_response}],
    context="수학 문제 - 검증 단계"
)

step_print(4, "다음 상태 (S')", f"히스토리 길이={len(next_state.history)}")

kv_print([
    ("상태 전이", f"S → S' (T(s'|s,a))"),
    ("결정적 여부", "Yes (temperature=0)"),
    ("히스토리 누적", f"{len(current_state.history)} → {len(next_state.history)}")
])
print("\
✓ 각 행동은 새로운 상태로의 전이를 야기한다.")


────────────────────────────────────────
  텍스트 상태 전이 시뮬레이션
────────────────────────────────────────
  [Step 1] 현재 상태
    2+2는 몇인가?
  [Step 2] 행동 (A)
    LLM 기반 응답 생성
  [Step 3] 응답
    2 + 2는 4입니다.
  [Step 4] 다음 상태 (S')
    히스토리 길이=1
  상태 전이    S → S' (T(s'|s,a))
  결정적 여부   Yes (temperature=0)
  히스토리 누적  0 → 1
✓ 각 행동은 새로운 상태로의 전이를 야기한다.


## 이론: POMDP (부분 관측 마르코프 의사결정 과정)

### 완전 관측 vs 부분 관측
- **MDP**: 에이전트가 현재 상태를 완전히 관찰할 수 있다
- **POMDP**: 에이전트가 상태를 부분적으로만 관찰할 수 있다

### LLM 에이전트는 왜 POMDP인가?
1. **제한된 맥락 윈도우**: 매우 긴 대화에서 전체 히스토리를 관찰할 수 없다
2. **숨겨진 사용자 의도**: 사용자가 말하지 않은 진정한 목표를 모른다
3. **동적 환경**: 실시간으로 변하는 외부 환경을 완전히 알 수 없다
4. **불확실한 도구 결과**: 도구 호출 결과가 예상과 다를 수 있다

### 신념 상태 (Belief State)
POMDP에서 에이전트는 가능한 모든 상태의 확률 분포를 유지한다.
b(s) = P(s | observations, actions)

## 이론: Generative Agent Memory와 MDP

### Memory Stream의 역할
Generative Agent의 Memory Stream이 MDP의 상태 공간을 정의한다:

**Memory Stream** → **관찰 함수** → **신념 상태** → **상태 표현**

1. **관찰 함수 O**: O(s) → observation
   - 환경의 숨겨진 상태 s에서 에이전트가 관찰할 수 있는 정보
   - 텍스트 형태의 관찰: \"사용자가 물었다: ...\"

2. **기억**: MemoryStream.add()로 관찰을 저장
   - Episodic memory: 구체적인 상호작용
   - Semantic memory: 추상적인 사실

3. **검색**: MemoryStream.retrieve()로 관련 기억 추출
   - 신념 상태의 일부를 형성

4. **행동 결정**: 신념 상태를 기반으로 정책 π(a|b) 적용

In [6]:
heading("MemoryStream과 POMDP 부분 관측")

# POMDP 에이전트: 메모리를 통해 신념 상태 유지
memory = MemoryStream()

# 관찰들을 메모리에 추가
observations = [
    "사용자: 2+2는 몇인가?",
    "에이전트: 2+2는 4입니다.",
    "환경: 사용자가 웃음소리를 냈다."
]

for i, obs in enumerate(observations):
    # [수정] timestamp 인자 제거 (내부에서 time.time() 자동 할당)
    memory.add(obs) 
    step_print(i+1, "관찰 추가", obs[:40] + "...")

# 신념 상태 형성을 위해 관련 기억 검색
belief_state = memory.retrieve("사용자 의도")
step_print(len(observations)+1, "신념 상태", f"{len(belief_state)} 개의 관련 기억 검색됨")

# 참고: 전체 메모리 크기를 확인할 때는 len(memory)를 사용하는 것이 더 정확합니다.
# retrieve("전체")는 "전체"라는 단어와 유사한 상위 k개만 가져옵니다.
kv_print([
    ("관찰 함수", "O(s) → 부분적 정보 제공"),
    ("메모리 크기", len(memory)), # [수정] retrieve 대신 len() 사용 권장
    ("신념 확률", "P(s | observations, actions)로 표현")
])

print("\n✓ POMDP에서 에이전트는 메모리를 통해 신념 상태를 유지한다.")


────────────────────────────────────────
  MemoryStream과 POMDP 부분 관측
────────────────────────────────────────
  [Step 1] 관찰 추가
    사용자: 2+2는 몇인가?...
  [Step 2] 관찰 추가
    에이전트: 2+2는 4입니다....
  [Step 3] 관찰 추가
    환경: 사용자가 웃음소리를 냈다....
  [Step 4] 신념 상태
    3 개의 관련 기억 검색됨
  관찰 함수   O(s) → 부분적 정보 제공
  메모리 크기  3
  신념 확률   P(s | observations, actions)로 표현

✓ POMDP에서 에이전트는 메모리를 통해 신념 상태를 유지한다.


## 실습 3: 부분 관측 환경에서 의사결정

In [7]:
heading("POMDP: 부분 관측 의사결정")

# 시나리오: 사용자의 진정한 의도는 숨겨져 있다
true_state = "사용자는 프로그래밍을 배우고 싶어한다"  # 숨겨진 상태
observed_text = "2+2는 몇인가?"  # 에이전트가 관찰한 것

step_print(1, "숨겨진 진정한 상태 (s)", "[에이전트는 모른다]")
step_print(2, "관찰 (O(s))", observed_text)

# 에이전트의 신념 상태 업데이트
system_prompt = "당신은 수학 튜터이다. 사용자의 진정한 의도를 파악하라."
belief_prompt = f"사용자가 '{observed_text}'라고 했다. 사용자의 진정한 의도는 무엇일까?"

belief_response = ask(
    belief_prompt, 
    system=system_prompt, 
    temperature=0.5
)
step_print(3, "신념 상태 (b)", belief_response[:80] + "...")

# 신념을 기반으로 행동 선택
action_prompt = f"신념: {belief_response}\
이를 바탕으로 어떻게 응답하겠는가?"
action = ask(action_prompt, temperature=0.5)
step_print(4, "선택된 행동 (π(a|b))", action[:80] + "...")

print("\
✓ POMDP 에이전트는 관찰에서 신념 상태를 추론하고 행동한다.")


────────────────────────────────────────
  POMDP: 부분 관측 의사결정
────────────────────────────────────────
  [Step 1] 숨겨진 진정한 상태 (s)
    [에이전트는 모른다]
  [Step 2] 관찰 (O(s))
    2+2는 몇인가?
  [Step 3] 신념 상태 (b)
    사용자가 '2+2는 몇인가?'라고 물은 것은 단순한 수학적 질문처럼 보이지만, 그 진정한 의도는 여러 가지일 수 있습니다. 사용자는 다음과 같은...
  [Step 4] 선택된 행동 (π(a|b))
    사용자의 질문에 대한 응답은 그 의도를 파악하여 적절하게 조절할 수 있습니다. 다음과 같은 방식으로 응답할 수 있습니다:
    
    1. **기본적인 수...
✓ POMDP 에이전트는 관찰에서 신념 상태를 추론하고 행동한다.


## 실습 4: 다단계 전이와 관측

In [12]:
heading("다단계 MDP 시뮬레이션")

memory = MemoryStream()
states = []
actions = []

# 초기 상태
initial_input = "내 이름이 뭐예요?"
states.append(initial_input)
memory.add(f"사용자: {initial_input}")

step_print(1, "초기 관찰", initial_input)

# 첫 번째 전이
action1 = ask(initial_input, temperature=0.0)
actions.append(action1)
memory.add(f"에이전트: {action1}")
step_print(2, "행동 a1", action1[:60] + "...")

# 사용자 피드백 (새로운 관찰)
feedback = "내 이름은 알렉스라고 했어요."
memory.add(f"사용자 피드백: {feedback}")
step_print(3, "새로운 관찰 O(s')", feedback)

# 두 번째 전이
correction_prompt = f"사용자가 '{feedback}'라고 했다. 이전 응답을 수정하라."
action2 = ask(correction_prompt, temperature=0.0)
actions.append(action2)
memory.add(f"에이전트: {action2}")
step_print(4, "행동 a2 (수정)", action2[:60] + "...")

# 최종 메모리 상태
final_memory = memory.retrieve("전체")
kv_print([
    ("총 단계 수", "2"),
    ("상태 전이", "s0 → s1 → s2"),
    ("메모리 항목", len(final_memory)),
    ("경로 의존성", "있음 (이전 상태가 현재 행동에 영향)")
])
print("\
✓ 다단계 상호작용에서 이전 상태는 이후 행동에 영향을 미친다.")


────────────────────────────────────────
  다단계 MDP 시뮬레이션
────────────────────────────────────────
  [Step 1] 초기 관찰
    내 이름이 뭐예요?
  [Step 2] 행동 a1
    죄송하지만, 당신의 이름을 알 수 있는 정보가 없습니다. 당신의 이름을 알려주시면 그에 맞춰 대화할 수 있습...
  [Step 3] 새로운 관찰 O(s')
    내 이름은 알렉스라고 했어요.
  [Step 4] 행동 a2 (수정)
    알겠습니다! 사용자가 "내 이름은 알렉스라고 했어요."라고 하셨다면, 이전 응답을 다음과 같이 수정할 수 있...
  총 단계 수  2
  상태 전이   s0 → s1 → s2
  메모리 항목  3
  경로 의존성  있음 (이전 상태가 현재 행동에 영향)
✓ 다단계 상호작용에서 이전 상태는 이후 행동에 영향을 미친다.


## 핵심 정리

| 개념 | 정의 | LLM 에이전트의 예 |
|------|------|-------------------|
| **S (상태)** | 관찰 가능한 정보 | 텍스트 입력, 히스토리, 맥락 |
| **A (행동)** | 에이전트의 선택 | 생성된 텍스트, 도구 호출 |
| **T (전이)** | 상태 변화 함수 | LLM의 조건부 분포 |
| **R (보상)** | 즉각적 피드백 | 사용자 평가, 작업 완료 신호 |
| **γ (할인)** | 미래 가치 감소율 | 대화가 길수록 초기 상태의 중요도 감소 |
| **POMDP** | 부분 관측 MDP | 맥락 윈도우, 숨겨진 의도 |
| **Memory** | 신념 상태 유지 | MemoryStream의 episodic/semantic memory |